In [15]:
import requests
from bs4 import BeautifulSoup
import re
import geopandas as gpd
from pathlib import Path

In [16]:
data_path = Path("C:/data")

# URL of the site to scrape
url = 'https://www.c2000masten.nl/'

# Send a GET request to the site
response = requests.get(url)

In [ ]:
# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content of the site
    soup = BeautifulSoup(response.content, 'html.parser')

else:
    print(f"Failed to retrieve the webpage. Status code: {response.status_code}")


In [5]:
# Find the script tag containing the marker data
script = soup.find('script', text=re.compile(r'google\.maps\.Marker'))

# Extract all marker lines
marker_lines = re.findall(r"google\.maps\.Marker\(\{.*?\}\);", script.string, re.DOTALL)

scraped_data = []
for line in marker_lines:
    # Extract lat, lng, and title
    lat_lng_match = re.search(r'lat\s*:\s*([0-9.]+),\s*lng\s*:\s*([0-9.]+)', line)
    title_match = re.search(r"title\s*:\s*'([^']+)'", line)
    if lat_lng_match and title_match:
        lat = float(lat_lng_match.group(1))
        lon = float(lat_lng_match.group(2))
        title = title_match.group(1)
        # Extract date from title
        date_match = re.search(r'\(([\d\-]+)\)', title)
        date = date_match.group(1) if date_match else None
        scraped_data.append({'date': date, 'location': (lat, lon)})

scraped_data[0]

C:\Users\peregrin\AppData\Local\Temp\ipykernel_28080\61151704.py:2: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  script = soup.find('script', text=re.compile(r'google\.maps\.Marker'))


{'date': '2007-09-07', 'location': (52.17594, 5.29254)}

In [8]:
# Create gdf from the scraped data with 'location' as geometry	and 'date' as an attribute
gdf = gpd.GeoDataFrame(
    scraped_data,
    geometry=gpd.points_from_xy(
        [loc['location'][1] for loc in scraped_data],  # Longitude
        [loc['location'][0] for loc in scraped_data]   # Latitude
    ),
    crs="EPSG:4326"  # WGS84 coordinate system
)

In [14]:
# gdf.explore()

In [18]:
# Save as geojson with metadata
gdf['source'] = 'https://www.c2000masten.nl/'
gdf.to_file(data_path/r'raw\TelecomData\c2000masten.geojson', driver='GeoJSON')

Repeat for 5G

In [10]:
# Repeat for the 5G masts
url_5g = 'https://5g-masten.nl/'

# Send a GET request to the 5G masts site
response_5g = requests.get(url_5g)

In [ ]:
# Check if the request was successful
if response_5g.status_code == 200:
    # Parse the HTML content of the 5G masts site
    soup_5g = BeautifulSoup(response_5g.content, 'html.parser')
else:
    print(f"Failed to retrieve the 5G masts webpage. Status code: {response_5g.status_code}")   

In [ ]:
soup_5g

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN"
    "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">

<html xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta content="Via deze site kunt u inzicht krijgen in de LTE 4G 5G dekking en van de digitale zendmasten bij u in  de buurt zoals GSM, UMTS, C2000 en digitenne" name="description"/>
<meta content="LTE, uitrol, GSM, UMTS, 3G netwerk, dekking,4G, 5G, frequentie, hoogte, telecom providers, C2000, zendmasten,KPN,Vodafone,T-Mobile, odido, Ziggo, Tele2, Wifi, Hotspot, masten,2G, zenders, digitenne,locaties, vaste verbinding, Amsterdam, mobiele, straling, telefoon, overzicht, kaart, map, lijst" name="keywords"/>
<meta content="text/html; charset=utf-8" http-equiv="content-type"/>
<meta content="index, follow" name="robots"/>
<meta content="Van Zuilen Internet Strategie" name="author"/>
<title>4G LTE masten 5G; Dekking voor snel internet van de telecom providers?</title>
<script src="https://maps.googleapis.com/maps/a

In [ ]:
# Find the script tag containing the marker data for 5G masts
script_5g = soup_5g.find('script', text=re.compile(r'google\.maps\.Marker'))

# Extract all marker lines
marker_lines_5g = re.findall(r"google\.maps\.Marker\(\{.*?\}\);", script_5g.string, re.DOTALL)

scraped_data_5g = []
for line in marker_lines_5g:
    # Extract lat, lng, and title
    lat_lng_match = re.search(r'lat\s*:\s*([0-9.]+),\s*lng\s*:\s*([0-9.]+)', line)
    title_match = re.search(r"title\s*:\s*'([^']+)'", line)
    if lat_lng_match and title_match:
        lat = float(lat_lng_match.group(1))
        lon = float(lat_lng_match.group(2))
        title = title_match.group(1)
        # Extract date from title
        date_match = re.search(r'\(([\d\-]+)\)', title)
        date = date_match.group(1) if date_match else None
        scraped_data_5g.append({'date': date, 'location': (lat, lon)})

scraped_data_5g[0]

C:\Users\peregrin\AppData\Local\Temp\ipykernel_28080\1597324385.py:2: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  script_5g = soup_5g.find('script', text=re.compile(r'google\.maps\.Marker'))


AttributeError: 'NoneType' object has no attribute 'string'